# ECLIPSE · Notebook 04 — XAI Demo

Explainability walkthrough for a TRANSIT candidate:
1. **Attention Rollout** — which raw flux patches drove the anomaly detection
2. **SHAP DeepExplainer** — global view phase bin attribution
3. **Captum Integrated Gradients** — local view transit morphology attribution
4. **Uncertainty Decomposition** — aleatoric vs epistemic
5. **PDF Report** — generate a complete per-candidate report

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.models.eclipse_prime import ECLIPSEPrime
from src.utils.config import DEFAULT_CONFIG
from src.utils.checkpoint import get_best_checkpoint, load_checkpoint
from src.inference.pipeline import ECLIPSEInferencePipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ECLIPSEPrime.from_config(DEFAULT_CONFIG).to(device)
ckpt = get_best_checkpoint(DEFAULT_CONFIG.api.checkpoint_dir)
if ckpt: load_checkpoint(ckpt, model, device=device)
model.eval()

TIC_ID = 261136679  # WASP-18b (known TRANSIT)
print(f'Running XAI demo on TIC {TIC_ID}')

In [ ]:
# Run full inference pipeline
pipe = ECLIPSEInferencePipeline(sector=1, config=DEFAULT_CONFIG)
result = pipe.run(TIC_ID)

print(f'Predicted class: {result.get("predicted_class", "N/A")}')
print(f'Confidence:      {result.get("confidence", 0):.1%}')
print(f'Period:          {result.get("period", 0):.4f} ± {result.get("period_err", 0):.4f} days')
print(f'Depth:           {result.get("depth", 0):.6f}')
print(f'TLS SDE:         {result.get("snr_tls", 0):.2f}')
if result.get('error'):
    print(f'Error: {result["error"]}')

In [ ]:
# Attention Rollout — which patches drove Stream A detection
from src.xai.attention_rollout import AttentionRollout
from src.ingestion.tess_fetcher import TESSFetcher
from src.preprocessing.denoising import full_denoising_pipeline, apply_quality_mask

fetcher = TESSFetcher(sector=1)
raw = fetcher.get_lightcurve(TIC_ID)

if raw:
    t, f, fe, cx, cy = apply_quality_mask(
        raw['time'], raw['flux'], raw.get('flux_err', np.ones_like(raw['flux'])*0.001),
        raw.get('quality', np.zeros(len(raw['time']), dtype=np.int16)),
        raw.get('centroid_x', np.zeros_like(raw['time'])),
        raw.get('centroid_y', np.zeros_like(raw['time']))
    )
    tc, fc, _ = full_denoising_pipeline(t, f, fe)

    rollout = AttentionRollout(model, device)
    importance = rollout.compute_rollout(fc)
    time_importance = rollout.importance_to_time(importance, len(fc))

    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    axes[0].plot(tc, fc, '.', ms=1, color='#4FC3F7', alpha=0.4)
    axes[0].set_ylabel('Flux'); axes[0].set_title('Detrended Flux')

    axes[1].fill_between(tc[:len(time_importance)], time_importance,
                         alpha=0.7, color='#CE93D8')
    axes[1].set_ylabel('Attention Importance')
    axes[1].set_xlabel('BTJD (days)')
    axes[1].set_title('Stream A Attention Rollout — High = anomalous flux region')

    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No raw data available for XAI demo.')

In [ ]:
# Batman model overlay on local view
from src.models.physics_loss import synthesize_batman_model
import numpy as np

if result.get('period') and result.get('duration_days') and result.get('depth'):
    batman_flux = synthesize_batman_model(
        period=result['period'],
        duration_days=result['duration_days'],
        depth=result['depth']
    )

    phase_l = np.linspace(-1, 1, len(batman_flux))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(phase_l, batman_flux - 1.0, 'r--', lw=2, label='batman model (predicted P,τ,δ)')
    ax.set_xlabel('Phase (transit durations)')
    ax.set_ylabel('Flux - 1.0')
    ax.set_title(f'batman Transit Model — P={result["period"]:.4f}d, δ={result["depth"]:.5f}')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No transit parameters predicted.')

In [ ]:
# Uncertainty decomposition with MC Dropout
from src.models.conformal import MCDropoutWrapper

if raw:
    mc = MCDropoutWrapper(model, n_samples=30)

    T_max = DEFAULT_CONFIG.model.T_max
    rf_pad = np.zeros(T_max, dtype=np.float32)
    rf_pad[:min(len(fc), T_max)] = fc[:T_max]

    rf_t = torch.from_numpy(rf_pad).unsqueeze(0).to(device)
    gv_t = torch.zeros(1, 2001).to(device)
    lv_t = torch.zeros(1, 201).to(device)
    sp_t = torch.zeros(1, 8).to(device)
    cv_t = torch.zeros(1, 201).to(device)

    unc = mc.epistemic_uncertainty(rf_t, gv_t, lv_t, sp_t, cv_t)
    print('Epistemic Uncertainty (MC Dropout, 30 samples):')
    print(f'  Mean class prob std:  {unc["prob_std"].mean():.4f}')
    print(f'  Period std:           {unc["period_std"].mean():.4f} days')
    print(f'  Duration std:         {unc["duration_std"].mean():.4f} days')
    print(f'  Depth std:            {unc["depth_std"].mean():.6f}')

In [ ]:
# Generate PDF report
from src.xai.pdf_reporter import PDFReporter

reporter = PDFReporter(output_dir='../data/reports')
pdf_path = reporter.generate(
    tic_id=TIC_ID,
    sector=1,
    result={**result, 'tic_id': TIC_ID, 'sector': 1},
    raw_time=tc if raw else None,
    raw_flux=fc if raw else None
)
if pdf_path:
    print(f'PDF report generated: {pdf_path}')
    print('Open in your PDF viewer to see the full report.')
else:
    print('PDF generation failed — check reportlab installation.')